# Python for AI — Class 20
### Machine Learning, part 3: Decision Trees, Random Forest, and checking a model properly

Today we learn a model that thinks like a person: it asks **yes/no questions**, one after another. It is called a **Decision Tree**.

Then we learn the most important skill of the week: **checking a model properly**.
A model can have **95% accuracy and still be useless**. Today you will see how, and how to catch it.

## How we will learn today

| Step | What happens |
|------|--------------|
| **CONCEPT** | The teacher explains the idea. Short points. |
| **EXAMPLE** | The teacher types the code. **You type it too** and run it. |
| **QUICK CHECK** | You solve a small task alone. |
| **🧑‍🏫 CLASS PROBLEM** | At the end, we solve one real problem **together**, step by step. |
| **🏠 HOME PROBLEM** | You solve the **same kind of problem** alone, on new data. Bring it to the next class. |

The solutions are in a separate file.

**Tip:** Run a cell with `Shift + Enter`.

## New words today

Don't worry if they are new. We will explain each one with an example.

| Word | Meaning in simple words |
|------|-------------------------|
| **Decision Tree** | a model that asks yes/no questions, one after another, to reach an answer |
| **Depth** | how many questions in a row the tree can ask |
| **Overfitting** | the model memorises the training data, but fails on new data |
| **Random Forest** | many decision trees that vote together |
| **Imbalanced data** | one group is much smaller than the other (e.g. 6% fraud, 94% normal) |
| **Confusion matrix** | a 2×2 table: what the model said vs what was really true |
| **Recall** | of all the real "yes" cases, how many did the model catch? |
| **Precision** | of all the times the model said "yes", how many were really yes? |

# Hour 1: Decision Trees and Random Forest

## CONCEPT: A Decision Tree asks questions

Think of a bank manager deciding on a loan:

```
                    Income above 166,000?
                    /                   \
                 NO                      YES
                 /                         \
      Loan above 25 lakh?            Defaulted before?
        /          \                  /           \
      YES          NO               NO            YES
       |            |                |              |
    REJECT    (more questions)    APPROVE     (more questions)
```

- Each box asks **one yes/no question** about one feature.
- You follow the answers from the **top** down, until you reach an answer.
- The tree **learns the questions itself** from the training data. It picks the question that **separates** "approved" and "not approved" best.
- `max_depth` = the most questions in a row. `max_depth=3` → at most 3 questions.

**Good news:** you can **read** a tree and explain it to anyone. Not every model is that easy.

## Get the data

The same `loans.csv` as last class: 400 loan applications.
**Using Colab?** The CSV files are in the same folder as this notebook. In Colab, click the **folder icon** on the left → **upload** the CSV files first.

In [ ]:
# On your own laptop, install once:  pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="white")

loans = pd.read_csv("loans.csv")

In [ ]:
X = loans[["monthly_income", "loan_amount", "years_in_job", "has_defaulted_before"]]
y = loans["approved"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(len(X_train), "to learn,", len(X_test), "to test")

### EXAMPLE: Train a small tree and draw it

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print("Accuracy:", accuracy_score(y_test, tree.predict(X_test)))

In [ ]:
plt.figure(figsize=(18, 8))
plot_tree(tree, feature_names=X.columns, class_names=["not approved", "approved"],
          filled=True, rounded=True, fontsize=9)
plt.show()

**How to read one box:**

| Line in the box | Meaning |
|-----------------|---------|
| `monthly_income <= 166000.0` | the **question**. True → go **left**. False → go **right** |
| `gini = 0.49` | how **mixed** the box is. 0 = all the same answer. 0.5 = half and half |
| `samples = 320` | how many training people reached this box |
| `value = [140, 180]` | how many of them were [not approved, approved] |
| `class = approved` | the answer if we stop here (the bigger group) |

**The colour:** orange = mostly "not approved", blue = mostly "approved". A darker colour means the box is more sure.

The tree's **first question is about income**. So income is the most useful feature for this bank.

### ✏️ QUICK CHECK 1: Follow the tree by hand

Look at the tree drawing. A person has: income **Rs 100,000**, loan **30 lakh**, 3 years in the job, never defaulted.

Follow the questions from the top. What does the tree decide? Then check with `tree.predict(...)`.

Write your code in the cell below.

In [ ]:
# Write your code here

## CONCEPT: Overfitting: a deep tree memorises

What if we let the tree ask **as many questions as it wants**?

- It keeps asking until **every training person** is in a box alone. Then it is 100% right on the training data.
- But it has **memorised** the training people, including their strange cases. It did not learn the general rule.
- On **new** people (the test data), it does worse.

**Like a student** who memorised the answers to last year's paper: 100% on that paper, but they fail the real exam with new questions.

This is called **overfitting**. It is one of the biggest problems in ML.

### EXAMPLE: Train vs test accuracy for different depths

In [ ]:
depths = range(1, 11)
train_acc, test_acc = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, t.predict(X_train)))
    test_acc.append(accuracy_score(y_test, t.predict(X_test)))

plt.plot(depths, train_acc, marker="o", label="train (seen before)")
plt.plot(depths, test_acc, marker="o", label="test (new people)")
plt.title("Deeper tree: better on train, NOT better on test")
plt.xlabel("max_depth (number of questions)")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

**How to read it:**
- The **train** line keeps going up, to **100%** at depth 10. The tree memorised everybody.
- The **test** line does **not** go up. It stays around 75% to 80%, and at the end it goes **down**.
- The **gap** between the two lines is the overfitting. A big gap = the model memorised.

**Lesson:** keep the tree **small** (3 to 5 questions). Always look at the **test** score, never only the train score.

## CONCEPT: Random Forest: many trees vote

- One tree can make strange mistakes.
- A **Random Forest** trains **many trees** (by default, 100). Each tree sees a slightly **different random part** of the training data.
- For a new person, **all trees vote**. The most common answer wins.
- **Like asking 100 doctors** instead of 1: one doctor can be wrong, but the majority is usually right.
- `n_estimators` = the number of trees.

### EXAMPLE: Random Forest

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
print("Random Forest accuracy:", accuracy_score(y_test, forest.predict(X_test)))

About **0.78**, about the same as the small tree. In Class 19, Logistic Regression got **0.86** on this same test data.

**Lesson:** Random Forest is often strong, but **no model is always the best**. That's why we **try a few models and compare** them on the same test data.

### ✏️ QUICK CHECK 2: How many trees?

Train Random Forests with `n_estimators = 10, 100, 300`. Print the test accuracy for each.

Do more trees always help?

Write your code in the cell below.

In [ ]:
# Write your code here

# Hour 2: Checking a Model Properly

## CONCEPT: The accuracy trap

**A story:** a test for **dengue**. We check **20 people**. **4** of them really have dengue.

**Lazy Model A** says **"healthy"** for everybody. It never says "dengue".
- It is right for the 16 healthy people → accuracy = 16 / 20 = **80%**.
- But it **found 0 of the 4 sick people**. It is **useless**!

**Model B** says "dengue" for 5 people. 3 of them really have dengue, and 2 are healthy.
- Right: 3 sick + 14 healthy = 17 → accuracy = 17 / 20 = **85%**.
- It **found 3 of the 4** sick people. **Much more useful.**

80% vs 85% looks like a small difference. But one model is useless and the other is useful!

**The trap:** when one group is **small** (imbalanced data), a lazy model gets high accuracy by always guessing the big group. **So accuracy alone can lie.**

## CONCEPT: The confusion matrix

A 2×2 table. It shows **all 4 kinds of answers**, for Model B:

```
                         MODEL SAID: healthy      MODEL SAID: dengue
REALLY healthy (16)      14  ✅ correct           2  ⚠️ false alarm
REALLY dengue  (4)        1  ❌ missed!            3  ✅ caught
```

| Box | Name | Meaning |
|-----|------|---------|
| ✅ caught | True Positive | really sick, model said sick |
| ❌ missed | False Negative | really sick, model said healthy. **Dangerous!** |
| ⚠️ false alarm | False Positive | really healthy, model said sick. Annoying, extra tests |
| ✅ correct | True Negative | really healthy, model said healthy |

(Don't worry about the long names. "Caught, missed, false alarm" is enough.)

## CONCEPT: Recall and precision

Two questions, two scores:

**Recall: "Of all the sick people, how many did we catch?"**
```
recall = caught / (caught + missed) = 3 / (3 + 1) = 0.75   →  we caught 75% of the sick people
```

**Precision: "When the model said 'dengue', how often was it right?"**
```
precision = caught / (caught + false alarms) = 3 / (3 + 2) = 0.60   →  60% of the alarms were real
```

**Which one matters more? It depends on the cost of each mistake:**

| Problem | Worst mistake | So we care most about |
|---------|---------------|-----------------------|
| disease test, fraud, a bomb at the airport | **missing** a real case | **recall** |
| spam filter (moving a real email to spam is bad) | a **false alarm** | **precision** |
| a balanced problem, both mistakes cost the same | either | **accuracy** is OK |

### EXAMPLE: Calculate it with code (Model B)

In [ ]:
# 1 = dengue, 0 = healthy. 20 people
real    = [1, 1, 1, 1] + [0] * 16
model_b = [1, 1, 1, 0] + [1, 1] + [0] * 14     # caught 3, missed 1, 2 false alarms

print("Accuracy: ", accuracy_score(real, model_b))
print("Recall:   ", recall_score(real, model_b))
print("Precision:", precision_score(real, model_b))
print(confusion_matrix(real, model_b))

`confusion_matrix` prints the same table:
```
[[14  2]      ← really healthy: 14 correct, 2 false alarms
 [ 1  3]]     ← really sick:     1 missed,  3 caught
```

### ✏️ QUICK CHECK 3: The lazy model

Make the **lazy Model A**: it says `0` (healthy) for all 20 people. (Hint: `[0] * 20`.)

Print its accuracy, recall and confusion matrix. What does the recall tell you?

Write your code in the cell below.

In [ ]:
# Write your code here

## CONCEPT: Checking a regression model: MAE and MSE

For **numbers** (like house prices), we don't use accuracy. We use the **error**, like in Class 18.

- **MAE** (Mean Absolute Error) = the average size of the errors.
- **MSE** (Mean Squared Error) = square each error (error × error) first, then take the average.

**Why square?** It makes **big mistakes count much more.** Errors of 2, 2 and 10 lakh:

```
MAE = (2 + 2 + 10) / 3          = 4.7
MSE = (4 + 4 + 100) / 3         = 36      ← the 10 became 100!
```

- Use **MAE** to **explain** the error to people ("wrong by 4.7 lakh on average").
- **MSE** is used when **one big mistake is very bad**. Models also learn with it (you will see it again with neural networks).

# Hour 3: Evaluate Properly, Not Just "Did It Run?"

## CONCEPT: A checklist for every model

Before you say "my model is good", check:

1. ✅ Did I test it on data it has **never seen** (the test data)?
2. ✅ Is the data **balanced**? If not, accuracy can lie. Look at **recall** and **precision**.
3. ✅ What does a **lazy model** get (always guessing the biggest group)? My model must be **clearly better**.
4. ✅ Look at the **confusion matrix**. Which mistakes is it making?
5. ✅ Which mistake is **worse** for this problem? Choose the score that matches it.
6. ✅ Is the **train** score much higher than the **test** score? Then it may be **overfitting**.

## CONCEPT: Make the model care about the small group

- With imbalanced data, the model sees "normal" so often that it learns to almost always say "normal".
- `class_weight="balanced"` tells the model: **"the small group is rare, but it is important. Pay more attention to it."**
- The model then catches **more** of the small group (**recall goes up**), but it also makes **more false alarms** (precision goes down).
- We will try it in the Class Problem.

# 🧑‍🏫 Class Problem: Catching card fraud

**The problem:** A bank has card transactions. A few of them are **fraud** (someone stole the card). The bank asks:
> "Can a model flag the fraud, so we can block those cards?"

`transactions.csv` has **2,000 transactions**:

| Column | Meaning |
|--------|---------|
| `amount` | the payment, in thousand Rs |
| `hour` | the hour of the day (0 = midnight, 23 = 11 pm) |
| `distance_from_home_km` | how far from the card owner's home the payment was made |
| `online` | 1 = online payment, 0 = in a shop |
| `fraud` | **the target**: 1 = fraud, 0 = normal |

We solve it **together**. Watch how accuracy tries to trick us.

### Step 1: Load and check the balance

In [ ]:
tx = pd.read_csv("transactions.csv")
print(tx.shape)
print(tx["fraud"].value_counts())
print("Fraud:", round(tx["fraud"].mean() * 100, 1), "%")

Only about **6%** are fraud. The data is **very imbalanced**. Warning: accuracy may lie!

### Step 2: Split (with `stratify`, so the test data also has about 6% fraud)

In [ ]:
X = tx[["amount", "hour", "distance_from_home_km", "online"]]
y = tx["fraud"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Frauds in the test data:", y_test.sum(), "of", len(y_test))

### Step 3: The lazy model (always says "normal")

In [ ]:
lazy = [0] * len(y_test)
print("Lazy accuracy:", accuracy_score(y_test, lazy))
print("Lazy recall:  ", recall_score(y_test, lazy))

**94% accuracy, and it catches 0 frauds!** Any real model must do better than this.

### Step 4: Train a decision tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
pred = tree.predict(X_test)

print("Accuracy: ", round(accuracy_score(y_test, pred), 3))
print("Recall:   ", round(recall_score(y_test, pred), 2))
print("Precision:", round(precision_score(y_test, pred), 2))

### Step 5: Look at the confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["normal", "fraud"], cmap="Blues")
plt.title("Decision tree: what it said vs the truth")
plt.show()

**95% accuracy sounds great. But look at the bottom row:**
- The test data has **24** frauds. The tree **caught only 6**, and **missed 18**!
- Recall = 6 / 24 = **0.25**. It lets 3 out of 4 frauds pass. For a bank, this is **bad**.
- It only beats the lazy model by about 1% accuracy.

### Step 6: Improve: make the tree care about fraud

In [ ]:
tree_bal = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_bal.fit(X_train, y_train)
pred_bal = tree_bal.predict(X_test)

print("Accuracy: ", round(accuracy_score(y_test, pred_bal), 3))
print("Recall:   ", round(recall_score(y_test, pred_bal), 2))
print("Precision:", round(precision_score(y_test, pred_bal), 2))

ConfusionMatrixDisplay.from_predictions(y_test, pred_bal, display_labels=["normal", "fraud"], cmap="Blues")
plt.title("Balanced tree")
plt.show()

### Step 7: Compare and decide

In [ ]:
compare = pd.DataFrame({
    "accuracy":  [accuracy_score(y_test, lazy), accuracy_score(y_test, pred), accuracy_score(y_test, pred_bal)],
    "recall":    [recall_score(y_test, lazy), recall_score(y_test, pred), recall_score(y_test, pred_bal)],
}, index=["lazy model", "tree", "balanced tree"])
compare.round(2)

**The surprise:** the balanced tree has **lower accuracy** (89% instead of 95%), but it is **much better** for the bank:

| | Tree | Balanced tree |
|--|------|---------------|
| Frauds caught (of 24) | 6 | **19** |
| Frauds missed | 18 | **5** |
| False alarms | 1 | 39 |

**Our answer to the bank:**
> "Use the balanced tree. It catches about **4 out of 5** frauds (recall 0.79), instead of 1 out of 4. It also blocks about 39 normal payments by mistake, so the bank should send an SMS to check them, not block the card forever. **Accuracy alone would have chosen the wrong model.**"

# 🏠 Home Problem: Which customers will leave?

A mobile network company is losing customers. It wants to find customers who may **leave**, and give them a special offer first. **Same kind of problem: a small group that matters.**

The data is in `churn.csv` (1,000 customers):

| Column | Meaning |
|--------|---------|
| `months_with_us` | how many months they have been a customer |
| `monthly_bill` | their bill each month (Rs) |
| `complaints` | how many complaints they made |
| `dropped_calls` | how many calls dropped last month |
| `left` | **the target**: 1 = left the company, 0 = stayed |

**Do the same 7 steps as the Class Problem:**

1. Load `churn.csv`. What % of customers left? Is it balanced?
2. Make `X` and `y`. Split with `test_size=0.2, random_state=42, stratify=y`. How many "left" are in the test data?
3. Find the accuracy and recall of the **lazy model** (always says 0).
4. Train `DecisionTreeClassifier(max_depth=3, random_state=42)`. Print accuracy, recall and precision.
5. Draw the confusion matrix. How many leavers did it catch, and how many did it miss?
6. Train it again with `class_weight="balanced"`. Print the 3 scores and the confusion matrix.
7. Make the comparison table, and choose a model.

**Bring to the next class:** your notebook, and 2 or 3 sentences for the company. Which score matters most here: recall or precision? Why?

In [ ]:
# Write your code here

# Home Practice: Interview-Style Questions

1. Explain a decision tree to someone who knows nothing about ML.
2. What is overfitting? How can you see it with train and test scores?
3. What is a Random Forest? Why can many trees be better than one?
4. A model has 95% accuracy on fraud data. Is it a good model? What would you check?
5. What is the difference between precision and recall? Give one problem where each one matters more.
6. What does `class_weight="balanced"` do? What do you gain, and what do you lose?
7. What is the difference between MAE and MSE?

# Key Takeaways

- A **Decision Tree** asks yes/no questions. You can draw it and read it.
- A deep tree **overfits**: 100% on train, worse on test. Keep it small with `max_depth`.
- A **Random Forest** = many trees that vote. Often strong, but not always the best. **Compare.**
- **Accuracy can lie** on imbalanced data. Always compare with the **lazy model**.
- The **confusion matrix** shows caught, missed and false alarms.
- **Recall** = how many real cases we caught. **Precision** = how many alarms were real.
- Choose the score by asking: **which mistake is worse?**
- `class_weight="balanced"` helps catch the small group.

**Next class:** Neural Networks: the idea behind modern AI, step by step.